In [3]:
import os
import nibabel as nib
import numpy as np 
from collections import OrderedDict
import json
from pathlib import Path
from scipy.ndimage import zoom
from tabulate import tabulate

from utils.helper import convert_nrrd_to_nifti, create_folder, plot_all_slices, plot_histogram, generate_binary_image, adjust_affine_for_spacing_and_origin, save_binary_image_with_adjusted_origin, make_if_dont_exist
from utils.metrics import dice_score_per_class, hausdorff_distance_per_class, ravd_per_class

### define project name and dataset sl

In [4]:
project_name = 'HCFC1' 
task_name = 'Dataset002_' + project_name 

### define and create dataset path

In [5]:
BASE_PATH = Path('./').resolve()
DATA_PATH = BASE_PATH / 'dataset'

TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTr'
GT_TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTr'
TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTs'
GT_TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTs'
PREDICTION_RESULTS_PATH  = BASE_PATH / 'dataset/nnUNet_Prediction_Results' / task_name
TASK_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name 

# setup environment variables
nnUNet_raw = BASE_PATH / 'dataset/nnUNet_raw_data'
nnUNet_preprocessed = BASE_PATH / 'dataset/nnUNet_preprocessed'
nnUNet_results = BASE_PATH / 'dataset/nnUNet_results'

make_if_dont_exist(TRAINING_DATASET_PATH,overwrite=False)
make_if_dont_exist(GT_TRAINING_DATASET_PATH)
make_if_dont_exist(TEST_DATASET_PATH)
make_if_dont_exist(GT_TEST_DATASET_PATH)
make_if_dont_exist(PREDICTION_RESULTS_PATH)

make_if_dont_exist(nnUNet_preprocessed)
make_if_dont_exist(nnUNet_results)

/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/imagesTr exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/imagesTs exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTs exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset002_HCFC1 exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_preprocessed exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results exists.


In [6]:
def convert_file_format(ORG_DATA_PATH, OUT_DATA_PATH, endswith):
    all_files = os.listdir(ORG_DATA_PATH)
    for file in all_files:
        if file.endswith(endswith):
            source_nrrd_file_path = os.path.join(ORG_DATA_PATH, file)
            save_nifti_file_path = os.path.join(OUT_DATA_PATH, file)
            # Automatically generate the nifti_file_path based on nrrd_file_path
            base_name = os.path.splitext(os.path.basename(save_nifti_file_path))[0]
            # print(base_name)
            newname = os.path.splitext(os.path.basename(base_name))[0]
            # print(newname)
            save_nifti_file_path = os.path.join(os.path.dirname(save_nifti_file_path), newname + '_0000.nii.gz')
            # print(source_nrrd_file_path)
            # print(save_nifti_file_path)
            convert_nrrd_to_nifti(source_nrrd_file_path, save_nifti_file_path)
    print('File convert done. ')

def label_downsample(img_data, label_data):
        
    # Get the shapes of the image and label data
    img_shape = img_data.shape
    label_shape = label_data.shape

    # Calculate the resize factor for each dimension
    resize_factors = [img_shape[i] / label_shape[i] for i in range(3)]

    # Resize the label data using scipy's zoom function
    resized_label_data = zoom(label_data, resize_factors, order=0)  
    return resized_label_data

def load_nifti_file(filepath):
    nifti_img = nib.load(filepath)
    return nifti_img.get_fdata(), nifti_img.shape

def save_nifti_file(data, filepath):
    nifti_img = nib.Nifti1Image(data.astype(np.float32), affine=np.eye(4))
    nib.save(nifti_img, filepath)

def rename_file_name(DATASET_PATH,remove_str, replace_str=""):
    # Iterate over the files in the directory
    for filename in os.listdir(DATASET_PATH):
        # Check if the file is a NIfTI image
        if filename.endswith(".nii.gz"):
            print(filename)
            # Extract the required part of the filename
            new_filename = filename.replace(remove_str,replace_str)
            # Construct the new path for the renamed file
            old_path = os.path.join(DATASET_PATH, filename)
            new_path = os.path.join(DATASET_PATH, new_filename)
            # Rename the file
            os.rename(old_path, new_path)
            
def file_compare(input_path,resize=False):
    # over view original Images 
    all_files = os.listdir(input_path)
    for file in all_files:
        if file.endswith(".nii.gz"):
            _label_path = os.path.join(input_path, file)

            file_id = file.split("_")
            file_name = f"{file_id[0]}_RCL5_0000.nii.gz"

            _file_path = os.path.join(TRAINING_DATASET_PATH, file_name)
            
            org_img, org_shape = load_nifti_file(_file_path)
            label_img, label_shape = load_nifti_file(_label_path)
            print(file_name)
            print("img: ",org_shape)
            print("label: ",label_shape)
            
            if org_shape != label_shape: 
                
                if resize:
                    resize_label = label_downsample(org_img, label_img)
                    # Save resized label
                    resized_label_filepath = os.path.join(input_path, file_name)  # Save with the same filename
                    save_nifti_file(resize_label, resized_label_filepath)

                    print("resize shape: ",resize_label.shape)
            else: 
                print("checked")
                
            print("")
    print('File read done. ')

def modify_intensity(input_filepath, output_filepath, intensity_map):
    # Load the NIfTI image
    nifti_img = nib.load(input_filepath)
    img_data = nifti_img.get_fdata()

    # Modify intensity values based on the provided intensity map
    modified_img_data = np.copy(img_data)
    for original_intensity, new_intensity in intensity_map.items():
        modified_img_data[img_data == original_intensity] = new_intensity

    # Save the modified image to a new NIfTI file
    modified_nifti_img = nib.Nifti1Image(modified_img_data, nifti_img.affine)
    nib.save(modified_nifti_img, output_filepath)
    return modified_img_data
    

def list_of_int(img_data):
    # Flatten the image data array to a 1D array
    flat_img_data = img_data.flatten()

    # Get the unique intensity values
    unique_values = set(flat_img_data)
    init_list = [int(x) for x in unique_values]

    return init_list

def check_intensity(path, modify_init=False):
    all_data = []
    intensity_map = {0: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9, 
                    11: 10, 12: 11, 13: 12, 14: 13, 15: 14, 16: 15, 17: 16, 18: 17, 
                    19: 18, 20: 19, 21: 20, 22: 21, 23: 22, 24: 23, 25: 24}

    all_files = os.listdir(path)
    for file in all_files:
        _label_path = os.path.join(path, file)
        img_data,_ = load_nifti_file(_label_path)
        new_filename = file.split("_")[0]  # Split the filename by underscore and get the first part
        # Check if all numbers from 0 to 24 are present in the array
        unique_value = list_of_int(img_data)
        result = all(i in unique_value for i in range(25))
        unq = {"Filename": new_filename, "Unique Intensity Values": unique_value,"status":result}
        all_data.append(unq)
        if modify_init and result == False: 
            modify = modify_intensity(_label_path, _label_path, intensity_map)
            unq = {"Filename": str(new_filename), "Unique Intensity Values": list_of_int(modify),"status":"modify"}
            all_data.append(unq)

    print(tabulate(all_data, headers="keys", tablefmt="grid"))


In [5]:
org_path = '/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/'

endswith = '.nrrd'
convert_file_format(org_path,org_path,endswith)

File convert done. 


### train img format convert 

In [102]:
org_path = '/work/shared/ngmm/3Dimage/DL_test/NewBrains_RCL5/'

endswith = '_masked.nrrd'
convert_file_format(org_path,TRAINING_DATASET_PATH,endswith)

File convert done. 


### train gt format convert

In [103]:
gt_path = '/work/shared/ngmm/3Dimage/DL_test/NewBrains_RCL5/'

endswith = 'Segments.seg.nrrd'
convert_file_format(gt_path,GT_TRAINING_DATASET_PATH,endswith)

File convert done. 


In [7]:
## test for NG4118
gt_path = '/work/shared/ngmm/3Dimage/DL_test/NewBrains_RCL5/'

endswith = 'Segments.seg.nrrd'
convert_file_format(gt_path,GT_TEST_DATASET_PATH,endswith)

File convert done. 


### Rename train file name

In [104]:
### Rename train file name
rename_file_name(TRAINING_DATASET_PATH,"_masked")
### Rename gt file name 
rename_file_name(GT_TRAINING_DATASET_PATH,"Segments_0000","RCL5")

NG4110_RCL5_masked_0000.nii.gz
NG4108_RCL5_masked_0000.nii.gz
NG4109_RCL5_masked_0000.nii.gz
NG4111_RCL5_masked_0000.nii.gz
NG4112_RCL5_masked_0000.nii.gz
NG4115_RCL5_masked_0000.nii.gz
NG4114_RCL5_masked_0000.nii.gz
NG4116_RCL5_masked_0000.nii.gz
NG4113_RCL5_masked_0000.nii.gz
NG4117_RCL5_masked_0000.nii.gz
NG4118_RCL5_masked_0000.nii.gz
NG4120_RCL5_masked_0000.nii.gz
NG4119_RCL5_masked_0000.nii.gz
NG4110_Segments_0000.nii.gz
NG4108_Segments_0000.nii.gz
NG4109_Segments_0000.nii.gz
NG4111_Segments_0000.nii.gz
NG4112_Segments_0000.nii.gz
NG4115_Segments_0000.nii.gz
NG4113_Segments_0000.nii.gz
NG4117_Segments_0000.nii.gz
NG4118_Segments_0000.nii.gz
NG4120_Segments_0000.nii.gz
NG4114_Segments_0000.nii.gz
NG4116_Segments_0000.nii.gz
NG4119_Segments_0000.nii.gz


In [6]:
file_compare(GT_TRAINING_DATASET_PATH,False)

NG4110_RCL5_0000.nii.gz
img:  (355, 898, 457)
label:  (355, 898, 457)
checked

NG4108_RCL5_0000.nii.gz
img:  (453, 789, 402)
label:  (453, 789, 402)
checked

NG4109_RCL5_0000.nii.gz
img:  (352, 761, 399)
label:  (352, 761, 399)
checked

NG4111_RCL5_0000.nii.gz
img:  (239, 879, 455)
label:  (239, 879, 455)
checked

NG4112_RCL5_0000.nii.gz
img:  (280, 794, 416)
label:  (280, 794, 416)
checked

NG4115_RCL5_0000.nii.gz
img:  (341, 822, 414)
label:  (341, 822, 414)
checked

NG4113_RCL5_0000.nii.gz
img:  (339, 856, 439)
label:  (339, 856, 439)
checked

NG4117_RCL5_0000.nii.gz
img:  (303, 797, 396)
label:  (303, 797, 396)
checked

NG4120_RCL5_0000.nii.gz
img:  (361, 780, 421)
label:  (361, 780, 421)
checked

NG4114_RCL5_0000.nii.gz
img:  (299, 754, 399)
label:  (299, 754, 399)
checked

NG4116_RCL5_0000.nii.gz
img:  (366, 939, 405)
label:  (366, 939, 405)
checked

NG4119_RCL5_0000.nii.gz
img:  (356, 745, 369)
label:  (356, 745, 369)
checked

File read done. 


In [7]:
train_files = os.listdir(TRAINING_DATASET_PATH)
label_files = os.listdir(GT_TRAINING_DATASET_PATH)
print("train image files:",len(train_files))
print("train label files:",len(label_files))
print("Matches:",len(set(train_files).intersection(set(label_files))))


train image files: 12
train label files: 12
Matches: 0


In [9]:
check_intensity(GT_TEST_DATASET_PATH)

+------------+------------------------------------------------------------------------------------------------------+----------+
| Filename   | Unique Intensity Values                                                                              | status   |
+============+======================================================================================================+==========+
| NG4118     | [0, 1, 8, 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 29, 30, 31, 32, 33, 34, 35, 36, 37] | False    |
+------------+------------------------------------------------------------------------------------------------------+----------+


In [113]:
check_intensity(GT_TRAINING_DATASET_PATH)

+------------+------------------------------------------------------------------------------------------------------+----------+
| Filename   | Unique Intensity Values                                                                              | status   |
+============+======================================================================================================+==========+
| NG4110     | [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]           | True     |
+------------+------------------------------------------------------------------------------------------------------+----------+
| NG4108     | [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]           | True     |
+------------+------------------------------------------------------------------------------------------------------+----------+
| NG4109     | [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22,

In [8]:
import nrrd
import pandas as pd
import matplotlib.pyplot as plt

def display_slice(img_data, slice_no=None):
    if slice_no is None:
        slice_no = img_data.shape[2] // 2  # Mid-slice if not specified
    plt.imshow(img_data[:, :, slice_no].T, cmap="gray", origin="lower")
    plt.title(f"Shape {img_data.shape}, Slice: {slice_no}" )
    plt.axis('off')
    plt.show()

def load_nrrd_file(nrrd_path):
    data, header = nrrd.read(nrrd_path)
    return data, header

def segment_label(nrrd_path, plot):
    data, header = load_nrrd_file(nrrd_path)
    filename = os.path.basename(nrrd_path)
    new_filename = filename.split("_")[0]
    segments = {}
    
    for key, value in header.items():
        if key.startswith("Segment") and key.endswith("_Name"):
            segment_number = int(''.join(filter(str.isdigit, key)))
            if 0 <= segment_number <= 24:
                segments[segment_number] = value
    if plot:
        display_slice(data)
    segments = dict(sorted(segments.items()))
    segments['file'] = new_filename
    return segments

def seg_file_list(ORG_DATA_PATH):
    all_files = os.listdir(ORG_DATA_PATH)
    seg_data = []
    for file in all_files:
        if file.endswith("Segments.seg.nrrd"):
            file_path = os.path.join(ORG_DATA_PATH, file)
            headerdata = segment_label(file_path, plot=False)
    
            # Append the file name and its segments as a single entry
            seg_data.append(headerdata)
    
    # # Convert to DataFrame
    # df = pd.DataFrame(seg_data)
    # Save to Excel
    # excel_path = os.path.join('/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/data', 'segmented_data.xlsx')
    # df.to_excel(excel_path, index=False)
    return seg_data
    


In [12]:
nrrd_path = '/work/shared/ngmm/3Dimage/DL_test/NewBrains_RCL5/'
data = seg_file_list(nrrd_path)
print(tabulate(data,headers="keys", tablefmt="grid"))

+------+------+-----+-----+-----+-----+-----+-----+-----+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+--------+
| 0    | 1    | 2   | 3   | 4   | 5   | 6   | 7   | 8   | 9   | 10   | 11   | 12   | 13   | 14   | 15   | 16   | 17   | 18   | 19   | 20   | 21   | 22   | 23   | 24   | file   |
+======+======+=====+=====+=====+=====+=====+=====+=====+=====+======+======+======+======+======+======+======+======+======+======+======+======+======+======+======+========+
| mask | CTX+ | cc+ | CPu | DG  | HP  | RHP | A   | ig  | fi  | f    | st   | ic   | och  | ac   | fr   | Hb   | TH   | HY   | MB   | P    | MY   | TCB  | V    | OB   | NG4110 |
+------+------+-----+-----+-----+-----+-----+-----+-----+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+--------+
| mask | CTX+ | cc+ | CPu | DG  | HP  | RHP | A   | ig  | fi  | f    | st   | ic   | och  | ac   | fr   | Hb  

In [13]:
nrrd_path = '/work/shared/ngmm/3Dimage/DL_test/prediction_metadata/'
data = seg_file_list(nrrd_path)
print(tabulate(data,headers="keys", tablefmt="grid"))

NRRDError: Duplicate header field: dimension

In [14]:
labels = {}

# Iterate over the range of indices in the first row of data
for i in range(len(data[0])):
    if i < 25:
        # Append a dictionary where the label is the key and the index is the value
        if data[0][i] == 'mask':
            labels['background'] = i
        else:
            labels[data[0][i]] = i

print(labels)


{'background': 0, 'CTX+': 1, 'cc+': 2, 'CPu': 3, 'DG': 4, 'HP': 5, 'RHP': 6, 'A': 7, 'ig': 8, 'fi': 9, 'f': 10, 'st': 11, 'ic': 12, 'och': 13, 'ac': 14, 'fr': 15, 'Hb': 16, 'TH': 17, 'HY': 18, 'MB': 19, 'P': 20, 'MY': 21, 'TCB': 22, 'V': 23, 'OB': 24}


In [15]:
# Using dictionary comprehension
swapped_dict = {value: key for key, value in labels.items()}

print(swapped_dict)


{0: 'background', 1: 'CTX+', 2: 'cc+', 3: 'CPu', 4: 'DG', 5: 'HP', 6: 'RHP', 7: 'A', 8: 'ig', 9: 'fi', 10: 'f', 11: 'st', 12: 'ic', 13: 'och', 14: 'ac', 15: 'fr', 16: 'Hb', 17: 'TH', 18: 'HY', 19: 'MB', 20: 'P', 21: 'MY', 22: 'TCB', 23: 'V', 24: 'OB'}


In [16]:
from typing import Tuple
import json
from os.path import join

def save_json(data, file_path, sort_keys=False):

    with open(file_path, 'w') as f:
        json.dump(data, f, indent=4, sort_keys=sort_keys)

def generate_dataset_json(output_folder: str,
                          channel_names: dict,
                          labels: dict,
                          num_training_cases: int,
                          file_ending: str,
                          regions_class_order: Tuple[int, ...] = None,
                          dataset_name: str = None, reference: str = None, release: str = None, license: str = None,
                          description: str = None,
                          overwrite_image_reader_writer: str = None, **kwargs):
    
    has_regions: bool = any([isinstance(i, (tuple, list)) and len(i) > 1 for i in labels.values()])
    if has_regions:
        assert regions_class_order is not None, f"You have defined regions but regions_class_order is not set. " \
                                                f"You need that."
    # channel names need strings as keys
    keys = list(channel_names.keys())
    for k in keys:
        if not isinstance(k, str):
            channel_names[str(k)] = channel_names[k]
            del channel_names[k]

    # labels need ints as values
    for l in labels.keys():
        value = labels[l]
        if isinstance(value, (tuple, list)):
            value = tuple([int(i) for i in value])
            labels[l] = value
        else:
            labels[l] = int(labels[l])

    dataset_json = {
        'channel_names': channel_names,  # previously this was called 'modality'. I didn't like this so this is
        # channel_names now. Live with it.
        'labels': labels,
        'numTraining': num_training_cases,
        'file_ending': file_ending,
    }

    if dataset_name is not None:
        dataset_json['name'] = dataset_name
    if reference is not None:
        dataset_json['reference'] = reference
    if release is not None:
        dataset_json['release'] = release
    if license is not None:
        dataset_json['licence'] = license
    if description is not None:
        dataset_json['description'] = description
    if overwrite_image_reader_writer is not None:
        dataset_json['overwrite_image_reader_writer'] = overwrite_image_reader_writer
    if regions_class_order is not None:
        dataset_json['regions_class_order'] = regions_class_order

    dataset_json.update(kwargs)

    save_json(dataset_json, join(output_folder, 'dataset.json'), sort_keys=False)
    

# List all files in the training images and labels directories
image_files = os.listdir(TRAINING_DATASET_PATH)
label_files = os.listdir(GT_TRAINING_DATASET_PATH)
test_ids = os.listdir(TEST_DATASET_PATH)

channel_names = {"0": "microscopic"}
num_training_cases = len(image_files)  
file_ending = ".nii.gz"

generate_dataset_json(
    output_folder=TASK_PATH,
    channel_names=channel_names,
    labels=labels, 
    num_training_cases=num_training_cases,
    file_ending=file_ending,
    dataset_name="Mouse Brain Segmentation",
    description="Mouse Brain Segmentation",
    reference="",
    release="0.0",
    license="",
    training = [{'image': f"./imagesTr/{image_file}", 'label': f"./labelsTr/{label_file}"} 
            for image_file, label_file in zip(sorted(image_files), sorted(label_files))],
    test=["./imagesTs/%s" % (i[:i.find("_0000")] + '.nii.gz') for i in test_ids]
)


In [17]:
import matplotlib.pyplot as plt
import numpy as np
import os 
import ipywidgets as widgets
from IPython.display import display


def read_nifti(path):
    img = nib.load(path)

    return img.get_fdata(),img.shape

# Load the NIfTI file
nifti_path = "/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/data/NG2566_Segments_Elo2_0000.nii.gz"
data, shape = read_nifti(nifti_path)
print(shape)
nifti_data = data
title = f"All Region, {shape}"

def visualize_regions(slice_no):
    middle_slice = slice_no 
    # Ensure that nifti_data has at least 24 regions
    if len(nifti_data) < 24:
        print("Insufficient number of regions in nifti_data")
        return
    
    # Calculate the number of rows and columns needed to display 24 regions
    num_rows = 5
    num_cols = 6
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(18, 12))

    for i in range(num_rows):
        for j in range(num_cols):
            index = i * num_cols + j
            if (index < 25) and (index < len(nifti_data)):
                # Extract the middle slice of the 3D volume
                roi = nifti_data == index
                # middle_slice = roi.shape[2] // 2
                axes[i, j].imshow(roi[:, :, middle_slice], cmap='gray')

                axes[i, j].set_title(f"{index}.{swapped_dict[index]}")
                axes[i, j].axis('off')
            else:
                # Hide empty subplots
                axes[i, j].axis('off')
    # Add main title
  
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


(1258, 1528, 831)


In [18]:

# visualize_regions(data )
# Interactive widget for slice selection
slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

widgets.interactive(visualize_regions, slice_no=slice_slider)

interactive(children=(IntSlider(value=415, description='Slice', max=830), Output()), _dom_classes=('widget-int…

In [ ]:
# Iterate over the files in the directory
for filename in os.listdir(GT_TRAINING_DATASET_PATH):
    # Check if the file is a NIfTI image
    if filename.endswith(".nii.gz"):
        # Extract the required part of the filename
        path = os.path.join(GT_TRAINING_DATASET_PATH, filename)
        data, shape = read_nifti(path)
        print(filename)
        print(shape)
        new_filename = filename.split("_")[0]
        title = f"{new_filename}, {shape}"
        visualize_regions(data,title )
        

In [1]:
import ipywidgets as widgets
from IPython.display import display

path = "/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr/NG4114_RCL5.nii.gz"
data, _  = read_nifti(path)
main_title = f"NG4114, {_}"
def _slice_visualize_regions(slice_no):

    
    # Calculate the number of rows and columns needed to display 24 regions
    num_rows = 4
    num_cols = 6
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(18, 6))

    for i in range(num_rows):
        for j in range(num_cols):
            index = i * num_cols + j
            if index < len(data):
                # Extract the middle slice of the 3D volume
                roi = data == index
                middle_slice = roi.shape[2] // 2
                axes[i, j].imshow(roi[:, :, slice_no], cmap='gray')

                axes[i, j].set_title(f"Region {index}, Slice {slice_no}")
                axes[i, j].axis('off')
            else:
                # Hide empty subplots
                axes[i, j].axis('off')
        # Add main title
    plt.suptitle(main_title)
    plt.tight_layout()
    plt.show()

slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

widgets.interactive(_slice_visualize_regions, slice_no=slice_slider)

NameError: name 'read_nifti' is not defined

In [21]:
input_filepath = "/work/shared/ngmm/scripts/Taiabur/imagetests/NG2604_right_0000.seg.nrrd"
data = segment_label(input_filepath, plot=False)
print(tabulate(data,headers="keys", tablefmt="grid"))

+--------+
| file   |
+========+
| N      |
+--------+
| G      |
+--------+
| 2      |
+--------+
| 6      |
+--------+
| 0      |
+--------+
| 4      |
+--------+
